In [ ]:
# =========================
# YOLOv5 Marine Debris Detection
# =========================

# Install if needed
# !pip install ultralytics opencv-python matplotlib

from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import os

# --------------------------------
# Dataset structure (already prepared)
# dataset/
#   images/train
#   images/val
#   labels/train
#   labels/val
# --------------------------------

# Create dataset config file
data_yaml = """
path: dataset
train: images/train
val: images/val

names:
  0: debris
"""

with open("marine.yaml","w") as f:
    f.write(data_yaml)

# Load pretrained YOLOv5
model = YOLO("yolov5s.pt")

# Train model
model.train(
    data="marine.yaml",
    epochs=20,
    imgsz=640,
    batch=8
)

# Validate model
metrics = model.val()

print("Mean Average Precision (mAP):", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

# ---------------------------
# Inference on test image
# ---------------------------

results = model("test.jpg")

for r in results:
    img = r.plot()

    plt.figure(figsize=(8,6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("YOLO Marine Debris Detection")
    plt.show()

In [ ]:
# =====================================
# EfficientNet-B0 Cow Image Classification
# =====================================

# !pip install timm

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset transforms
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# Dataset structure
# cow_dataset/
#    train/
#    val/

train_data = torchvision.datasets.ImageFolder("cow_dataset/train", transform=transform)
val_data = torchvision.datasets.ImageFolder("cow_dataset/val", transform=transform)

train_loader = torch.utils.data.DataLoader(train_data,batch_size=32,shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data,batch_size=32)

num_classes = len(train_data.classes)

# Load EfficientNet
model = timm.create_model("efficientnet_b0", pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_acc=[]
val_acc=[]
losses=[]

# Training
for epoch in range(10):

    model.train()
    correct=0
    total=0
    running_loss=0

    for images,labels in train_loader:

        images,labels = images.to(device),labels.to(device)

        outputs=model(images)
        loss=criterion(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()

        _,pred=torch.max(outputs,1)

        total+=labels.size(0)
        correct+=(pred==labels).sum().item()

    train_accuracy = 100*correct/total
    train_acc.append(train_accuracy)
    losses.append(running_loss)

    # Validation
    model.eval()
    correct=0
    total=0

    with torch.no_grad():
        for images,labels in val_loader:

            images,labels = images.to(device),labels.to(device)

            outputs=model(images)
            _,pred=torch.max(outputs,1)

            total+=labels.size(0)
            correct+=(pred==labels).sum().item()

    val_accuracy = 100*correct/total
    val_acc.append(val_accuracy)

    print(f"Epoch {epoch+1} Train Acc:{train_accuracy:.2f} Val Acc:{val_accuracy:.2f}")

# Accuracy plot
plt.figure()
plt.plot(train_acc,label="Train Accuracy")
plt.plot(val_acc,label="Validation Accuracy")
plt.legend()
plt.title("Accuracy Curve")
plt.show()

# Loss plot
plt.figure()
plt.plot(losses)
plt.title("Loss Curve")
plt.show()

In [ ]:
# =====================================
# MobileNetV2 Pest Classification (IP102)
# =====================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import time
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# Dataset structure
# IP102/
#   train/
#   test/

train_data = torchvision.datasets.ImageFolder("IP102/train", transform=transform)
test_data = torchvision.datasets.ImageFolder("IP102/test", transform=transform)

train_loader = torch.utils.data.DataLoader(train_data,batch_size=32,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data,batch_size=32)

num_classes = len(train_data.classes)

# Load MobileNetV2
model = torchvision.models.mobilenet_v2(pretrained=True)

# Replace classifier layer
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)

# Training
for epoch in range(5):

    model.train()

    for images,labels in train_loader:

        images,labels = images.to(device),labels.to(device)

        outputs=model(images)
        loss=criterion(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:",epoch+1,"Loss:",loss.item())

# --------------------------
# Evaluation
# --------------------------

model.eval()

correct=0
total=0

with torch.no_grad():
    for images,labels in test_loader:

        images,labels = images.to(device),labels.to(device)

        outputs=model(images)
        _,pred=torch.max(outputs,1)

        total+=labels.size(0)
        correct+=(pred==labels).sum().item()

accuracy = 100*correct/total
print("Accuracy:",accuracy)

# --------------------------
# Model Size
# --------------------------

torch.save(model.state_dict(),"mobilenet.pth")

size = os.path.getsize("mobilenet.pth")/1e6
print("Model Size (MB):",size)

# --------------------------
# Inference Time
# --------------------------

start=time.time()

for images,_ in test_loader:
    images = images.to(device)
    outputs = model(images)
    break

end=time.time()

print("Inference Time:",end-start,"seconds")

# part b tensorflow

In [ ]:
# ==========================================
# Cow Image Classification using EfficientNetB0
# ==========================================

import tensorflow as tf
import matplotlib.pyplot as plt

# Dataset directory
train_dir = "cow_dataset/train"
val_dir = "cow_dataset/val"

IMG_SIZE = (224,224)
BATCH_SIZE = 32

# Load datasets
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

num_classes = len(train_ds.class_names)

# --------------------------------
# EfficientNet Transfer Learning
# --------------------------------

base_model = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128,activation='relu')(x)
output = tf.keras.layers.Dense(num_classes,activation='softmax')(x)

model = tf.keras.Model(inputs=base_model.input,outputs=output)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Training
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

# --------------------------------
# Accuracy Plot
# --------------------------------

plt.plot(history.history['accuracy'],label="Train Accuracy")
plt.plot(history.history['val_accuracy'],label="Validation Accuracy")
plt.legend()
plt.title("Accuracy Curve")
plt.show()

# --------------------------------
# Loss Plot
# --------------------------------

plt.plot(history.history['loss'],label="Training Loss")
plt.plot(history.history['val_loss'],label="Validation Loss")
plt.legend()
plt.title("Loss Curve")
plt.show()

# part c tensorflow

In [ ]:
# ==========================================
# Pest Classification using MobileNetV2
# ==========================================

import tensorflow as tf
import time
import os

IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_dir = "IP102/train"
test_dir = "IP102/test"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

num_classes = len(train_ds.class_names)

# --------------------------------
# MobileNetV2 Transfer Learning
# --------------------------------

base_model = tf.keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
output = tf.keras.layers.Dense(num_classes,activation='softmax')(x)

model = tf.keras.Model(inputs=base_model.input,outputs=output)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Training
history = model.fit(train_ds,epochs=5)

# --------------------------------
# Evaluation
# --------------------------------

loss,accuracy = model.evaluate(test_ds)

print("Test Accuracy:",accuracy)

# --------------------------------
# Model Size
# --------------------------------

model.save("mobilenet_model.h5")
size = os.path.getsize("mobilenet_model.h5")/1e6

print("Model Size (MB):",size)

# --------------------------------
# Inference Time
# --------------------------------

for images,_ in test_ds.take(1):
    start=time.time()
    preds=model.predict(images)
    end=time.time()

print("Inference Time:",end-start,"seconds")